# yfinance — Real Financial Data — Week 1
**Aayan Mulani │ Decimal Point Analytics Preparation**

yfinance is a Python library that downloads historical price data from Yahoo Finance
for any stock, index, or ETF. Instead of manually downloading CSV files, yfinance
lets us pull clean, ready-to-use data with a single line of code. This is the
standard starting point for any quantitative finance analysis in Python.

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

## 1. Downloading Data from Yahoo Finance

We start by downloading the Nifty 50 index — 3 years of daily price data.
The ticker for Nifty 50 on Yahoo Finance is ^NSEI.
yf.download() returns a DataFrame with columns: Open, High, Low, Close, Volume.

In [4]:
# Download Nifty 50 index — 3 years of daily data
nifty = yf.download('^NSEI', start='2022-01-01', end='2025-01-01')

print(nifty.shape)
print(nifty.head())
print(nifty.tail())

[*********************100%***********************]  1 of 1 completed

(739, 5)
Price              Close          High           Low          Open  Volume
Ticker             ^NSEI         ^NSEI         ^NSEI         ^NSEI   ^NSEI
Date                                                                      
2022-01-03  17625.699219  17646.650391  17383.300781  17387.150391  200500
2022-01-04  17805.250000  17827.599609  17593.550781  17681.400391  247400
2022-01-05  17925.250000  17944.699219  17748.849609  17820.099609  251500
2022-01-06  17745.900391  17797.949219  17655.550781  17768.500000  236500
2022-01-07  17812.699219  17905.000000  17704.550781  17797.599609  239300
Price              Close          High           Low          Open  Volume
Ticker             ^NSEI         ^NSEI         ^NSEI         ^NSEI   ^NSEI
Date                                                                      
2024-12-24  23727.650391  23867.650391  23685.150391  23769.099609  177700
2024-12-26  23750.199219  23854.500000  23653.599609  23775.800781  177700
2024-12-27  2381

## 2. Downloading 5 Indian Stocks at Once

yfinance can download multiple tickers in a single call by passing a list.
We extract only the 'Close' price column since that is what we need for return calculations.
The .NS suffix tells Yahoo Finance these stocks are listed on the National Stock Exchange of India.

In [17]:
# Download 5 Indian stocks at once
tickers = ['RELIANCE.NS', 'TCS.NS', 'INFY.NS', 'HDFCBANK.NS', 'ITC.NS']
stocks = yf.download(tickers, start = '2022-01-01', end = '2025-01-01')['Close']

print(stocks.shape)
print(stocks.head())
print(stocks.isnull().sum())

[*********************100%***********************]  5 of 5 completed

(739, 5)
Ticker      HDFCBANK.NS      INFY.NS      ITC.NS  RELIANCE.NS       TCS.NS
Date                                                                      
2022-01-03   722.802612  1704.249512  177.217117  1093.780518  3386.357910
2022-01-04   727.035767  1704.877930  178.147293  1118.464966  3445.787109
2022-01-05   744.301331  1655.953003  178.389969  1123.697754  3424.676270
2022-01-06   732.362915  1631.849487  176.893616  1099.536621  3377.221924
2022-01-07   737.499756  1628.707520  176.650940  1108.409180  3418.068115
Ticker
HDFCBANK.NS    0
INFY.NS        0
ITC.NS         0
RELIANCE.NS    0
TCS.NS         0
dtype: int64


## 3. Computing Daily Returns

pct_change() computes the percentage change between each row and the previous row.
This gives us the daily return for each stock.
We drop the first row with dropna() because Day 1 has no previous day to compare against.

In [31]:
# Compute daily percentage returns for all 5 stocks
returns = stocks.pct_change().dropna()

print(returns.head())
print(returns.describe().round(4))

Ticker      HDFCBANK.NS   INFY.NS    ITC.NS  RELIANCE.NS    TCS.NS
Date                                                              
2022-01-04     0.005857  0.000369  0.005249     0.022568  0.017550
2022-01-05     0.023748 -0.028697  0.001362     0.004679 -0.006127
2022-01-06    -0.016040 -0.014556 -0.008388    -0.021501 -0.013857
2022-01-07     0.007014 -0.001925 -0.001372     0.008069  0.012095
2022-01-10     0.005546  0.020090  0.022436     0.000821  0.006838
Ticker  HDFCBANK.NS   INFY.NS    ITC.NS  RELIANCE.NS    TCS.NS
count      738.0000  738.0000  738.0000     738.0000  738.0000
mean         0.0004    0.0002    0.0013       0.0002    0.0003
std          0.0138    0.0156    0.0121       0.0142    0.0132
min         -0.0844   -0.0942   -0.0398      -0.0749   -0.0542
25%         -0.0065   -0.0085   -0.0058      -0.0080   -0.0065
50%          0.0007    0.0002    0.0010       0.0003   -0.0001
75%          0.0070    0.0096    0.0086       0.0078    0.0072
max          0.1001    0.07

## 4. Cumulative Returns

Cumulative returns show how a Rs 1 investment in each stock grew over 3 years.
Formula: (1 + r1) × (1 + r2) × ... - 1
cumprod() multiplies all the (1 + daily_return) values row by row.
Subtracting 1 at the end converts the multiplier back into a return percentage.

In [34]:
# Cumulative returns — how a Rs 1 investment grew over 3 years
cumulative = (1 + returns).cumprod() - 1

print('Cumulative Return Over 3 Years:')
print((cumulative.iloc[-1] * 100).round(2).to_string())

Cumulative Return Over 3 Years:
Ticker
HDFCBANK.NS     21.00
INFY.NS          7.14
ITC.NS         148.81
RELIANCE.NS     10.68
TCS.NS          14.77


## 5. Annualised Metrics and Sharpe Ratio

Daily returns are too granular to compare stocks directly.
We annualise them using 252 — the number of trading days in a year.

Annualised Return = Daily Mean Return × 252

Annualised Volatility = Daily Std Dev × sqrt(252)

Sharpe Ratio = Annualised Return / Annualised Volatility

The Sharpe Ratio measures return per unit of risk.
A Sharpe above 1 is considered good. Higher is always better.
We use sqrt(252) for volatility because volatility scales with the
square root of time, not linearly like returns.

In [36]:
# Annualised metrics for all 5 stocks
ann_return = returns.mean() * 252
ann_volatility = returns.std() * (252 ** 0.5)
sharpe = ann_return / ann_volatility

metrics = pd.DataFrame({
    'Ann_Return_%': (ann_return * 100).round(2),
    'Ann_Volatility_%': (ann_volatility * 100).round(2),
    'Sharpe_Ratio': sharpe.round(3)
})

print(metrics)

             Ann_Return_%  Ann_Volatility_%  Sharpe_Ratio
Ticker                                                   
HDFCBANK.NS          8.90             21.87         0.407
INFY.NS              5.42             24.73         0.219
ITC.NS              32.97             19.14         1.722
RELIANCE.NS          6.02             22.61         0.266
TCS.NS               6.89             20.94         0.329


## 6. Key Takeaways

1. yfinance downloads clean historical price data for any stock or index in one line of code.
2. Daily returns are computed using pct_change() — percentage change from previous day.
3. Cumulative returns show total growth of a Rs 1 investment over the full period.
4. Annualised return = daily mean × 252 (trading days in a year).
5. Annualised volatility = daily std × sqrt(252) — volatility scales with square root of time.
6. Sharpe Ratio = annualised return / annualised volatility — measures return per unit of risk.

ITC was the standout performer (2022-2024) — highest return, lowest volatility, Sharpe of 1.72.
INFY was the worst — highest volatility with the lowest return, Sharpe of only 0.22.
A Sharpe above 1 is considered good. Always compare both return AND risk, never return alone.